In [13]:
!pip install transformers torch


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [16]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small")

print("Model loaded successfully")

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 7792.70it/s]


Model loaded successfully


In [ ]:
def chatbot():
    chat_history_ids = None

    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

    while True:
        user_input = input("User: ").strip()

        # Skip empty input
        if user_input == "":
            continue

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! 👋")
            break

        # Encode input
        new_input_ids = tokenizer.encode(
            user_input + tokenizer.eos_token,
            return_tensors='pt'
        )

        # Maintain history
        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        # Limit history (important)
        if bot_input_ids.shape[-1] > 500:
            bot_input_ids = bot_input_ids[:, -500:]

        # Attention mask
        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        # Generate response (balanced + no repetition)
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=200,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,          # allow variation
            top_k=50,
            top_p=0.95,
            temperature=0.6,
            no_repeat_ngram_size=3   # 🔥 prevents repeating lines
        )

        # Decode response
        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        # Handle empty/bad output
        if response.strip() == "":
            response = "Sorry, I didn't understand that. Can you try again?"

        print("Chatbot:", response)